# 02 · Chunking

**Goal:** compare chunking configurations on *your* corpus and inspect the results, before embedding anything.
Code lives in `app/ingestion/chunking.py`. The final choice between configs is made by the retrieval evaluation (hit@k / MRR), not by this notebook; this notebook shows what each config *produces*.

Configs: **A** 400 tokens / 50 overlap · **B** 800 / 100 · **C** 200 / 30

In [1]:
import sys, pathlib, statistics
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from app.config import get_settings
from app.errors import AppError
from app.ingestion.validate import validate_upload
from app.ingestion.extract import extract_text
from app.ingestion.chunking import chunk_document, get_token_counter

settings = get_settings()
counter = get_token_counter()
print("token counter:", counter.name)

docs = {}
for path in sorted((ROOT / "corpus").iterdir()):
    if path.suffix.lower() not in {".pdf", ".docx"}:
        continue
    data = path.read_bytes()
    try:
        up = validate_upload(path.name, data, settings.max_upload_mb * 1024 * 1024)
        docs[path.name] = extract_text(up.file_type, data, max_pages=settings.max_pdf_pages)
    except AppError as exc:
        print("skipped", path.name, "->", exc.message)
print(len(docs), "documents loaded")

token counter: tiktoken:cl100k_base
2 documents loaded


## 1. Configurations side by side

In [2]:
CONFIGS = {"A 400/50": (400, 50), "B 800/100": (800, 100), "C 200/30": (200, 30)}
results = {name: {d: chunk_document(doc, size=s, overlap=o, counter=counter) for d, doc in docs.items()}
           for name, (s, o) in CONFIGS.items()}

source_tokens = sum(counter(s.text) for doc in docs.values() for s in doc.sections)
print(f"source tokens in corpus: {source_tokens}\n")
print(f"{'config':10} {'chunks':>6} {'min':>5} {'median':>6} {'max':>5} {'<50 tok':>8} {'tokens embedded':>16} {'overhead':>9}")
for name, per_doc in results.items():
    chunks = [c for cs in per_doc.values() for c in cs]
    sizes = [c.token_count for c in chunks]
    small = 100 * sum(1 for t in sizes if t < 50) / len(sizes)
    total = sum(sizes)
    print(f"{name:10} {len(chunks):>6} {min(sizes):>5} {int(statistics.median(sizes)):>6} {max(sizes):>5} "
          f"{small:>7.0f}% {total:>16} {100*(total/source_tokens-1):>8.0f}%")

source tokens in corpus: 2169

config     chunks   min median   max  <50 tok  tokens embedded  overhead
A 400/50        9   109    313   395       0%             2396       10%
B 800/100       5   128    461   636       0%             2169        0%
C 200/30       15   108    178   199       0%             2604       20%


`overhead` is the extra embedding volume caused by overlap (and affects cost/latency).

## 2. Chunk-size distribution (config A)

In [3]:
sizes = [c.token_count for cs in results["A 400/50"].values() for c in cs]
width = max(sizes) // 10 + 1
buckets = {}
for t in sizes:
    buckets[t // width] = buckets.get(t // width, 0) + 1
for b in sorted(buckets):
    print(f"{b*width:>5}-{(b+1)*width-1:<5} {'#' * buckets[b]} {buckets[b]}")

   80-119   # 1
  120-159   ## 2
  160-199   # 1
  280-319   # 1
  360-399   #### 4


## 3. Boundaries and overlap (config A)
End of one chunk vs start of the next: do boundaries fall at sensible places, and is the overlap present?

In [4]:
name, cs = next((n, c) for n, c in results["A 400/50"].items() if len(c) > 2)
print("document:", name)
for a, b in zip(cs[:2], cs[1:3]):
    print(f"\n--- end of chunk {a.position} (page={a.page}, section={a.section}) ---\n...{a.text[-220:]}")
    print(f"--- start of chunk {b.position} (page={b.page}, section={b.section}) ---\n{b.text[:220]}...")

document: nimbus_faq.pdf

--- end of chunk 0 (page=1, section=None) ---
...ing. Express shipping is not available for PO boxes.

Q10. What is the cut-off time for same-day dispatch?

A: Orders placed before 2:00 PM Eastern Time on a business day are dispatched the same day.

Returns and Refunds
--- start of chunk 1 (page=1, section=None) ---
A: Yes, but only with standard shipping. Express shipping is not available for PO boxes.

Q10. What is the cut-off time for same-day dispatch?

A: Orders placed before 2:00 PM Eastern Time on a business day are dispatche...

--- end of chunk 1 (page=1, section=None) ---
...an return most items within 30 days of delivery.

Q12. What condition must a returned item be in?

A: Items must be unused, in their original packaging, with all tags attached.

Nimbus Outdoor Co. Customer FAQ  |  Page 1
--- start of chunk 2 (page=2, section=None) ---
Q13. Who pays for return shipping?

A: Return shipping is free if the item is defective or we sent the wrong item

## 4. Suspiciously small chunks
Tiny chunks (lone headings, short pages) embed poorly. Worth knowing how many there are.

In [5]:
for name, per_doc in results.items():
    tiny = [(d, c) for d, cs in per_doc.items() for c in cs if c.token_count < 30]
    print(f"{name}: {len(tiny)} chunk(s) under 30 tokens")
for d, c in [(d, c) for d, cs in results["A 400/50"].items() for c in cs if c.token_count < 30][:5]:
    print(f"  {d} pos={c.position} page={c.page} section={c.section!r}: {c.text[:90]!r}")

A 400/50: 0 chunk(s) under 30 tokens
B 800/100: 0 chunk(s) under 30 tokens
C 200/30: 0 chunk(s) under 30 tokens


## 5. Table integrity
Every serialised table row (`Header: value | Header: value`) should sit whole inside one chunk.

In [6]:
for name, per_doc in results.items():
    rows = broken = 0
    for d, doc in docs.items():
        texts = [c.text for c in per_doc[d]]
        for s in doc.sections:
            for line in s.text.split("\n"):
                if " | " in line:
                    rows += 1
                    if not any(line.strip() in t for t in texts):
                        broken += 1
    print(f"{name}: {rows} table rows, {broken} split across chunks")

A 400/50: 5 table rows, 0 split across chunks
B 800/100: 5 table rows, 0 split across chunks
C 200/30: 5 table rows, 0 split across chunks


## 6. Notes for DESIGN.md (fill in)
- Which config looks best by eye, and why: ...
- Chunk-size trade-off observed (small = precise but context-poor; large = context-rich but noisy): ...
- Overlap cost (overhead %): ...
- Anything odd (tiny chunks, broken tables, repeated headers): ...

**Decision rule:** keep A and B (and C if interesting) as the configs to compare in the retrieval evaluation; the winner is picked from hit@k / MRR.